<a href="https://colab.research.google.com/github/shown5/Hands-on-Generative-AI/blob/main/chap6_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

６章言語モデルのファインチューニング

- ファインチューニング済みエンコーダーモデルを用いたテキストのトピック分類
- 現代の LLM 時代におけるエンコーダーベースモデルの役割の理解
- デコーダーモデルを使った特定のスタイルのテキスト生成
- 命令型ファインチューニングによる単一モデルでの複数タスクの解決
- 小さいGPUでもモデルを訓練できるパラメーター効率の高いファインチューニング手法
- よりすくに計算資源でモデルの推論を実行できる手法

6.1.1 データセットの特定

In [13]:
%pip install genaibook

  Using cached transformers-5.4.0-py3-none-any.whl.metadata (32 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 888.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 4

In [14]:
from datasets import load_dataset

#az news データセットはテキスト分類モデルのベンチマークやデータマイニング、情報検索、データストリーミングなどの研究で広く用いられている。
# 訓練用のサンプルは１２万件あり、ファインにチューニングには十分
raw_datasets = load_dataset("fancyzhx/ag_news")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [15]:
# データの具体的な例を見ていこう

raw_train_datasets = raw_datasets["train"]
raw_train_datasets[0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

//{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

サンプルにテキストとラベルが含まれているが、２はどのクラスを指しているのか？
これを知るにはデータセットの features とその label フィールドを見ればいい。

In [16]:
print(raw_train_datasets.features)

# results
# {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}
# 0なら世界のニュース、1ならスポーツ、2ならビジネス、３なら科学技術のニュース

{'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}


# 6.1.2 使用するモデルタイプの定義

## Transformer おさらい

- エンコーダーモデル：入力の意味表現を捉える
- デコーダーモデル：文章などの新しいシーケンスを出力することを目的にしたモデル。テキスト生成に最適。
- エンコーダーデコーダー型モデル：入力シーケンスを異なる出力シーケンスに変換するタスクに適している

今回は、**分類ヘッド付きエンコーダーモデル**をアプローチとして採用する。
エンコーダーモデルにシンプルな分類ネットワーク（ヘッド）を埋め込みに追加してファインチューニングする方法。

ベースモデルの要件は以下の４つ

- エンコーダベースであること
- GPU を使えば数分くらいでファインチューニングできるモデル
- 事前訓練で確かな成果を残しているもの
- 短いテキストシーケンスを処理できること

DistilBERT が良さげらしい。

6.1.4 データセットの前処理

トークナイザーは AutoTokenizer を使おう。
transformers ライブラリは入力の長さがすべて同じ出なければならないので、padding=true で使おう。


In [17]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(batch):
  return tokenizer(
      batch["text"], truncation=True, padding=True, return_tensors="pt"
  )

tokenize_function(raw_train_datasets[:2])

ImportError: cannot import name 'is_flax_available' from 'transformers.utils.import_utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py)

In [ ]:
# データセット内の各要素に対して関数を並列で適用するもの

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

6.1.5 評価指標の定義

評価指標には evaluate というライブラリが使える。
文章分類の場合は以下の指標が有力な候補となる。

- 正解率
- 適合率
- 再現率
- F1 スコア

evaluate が提供する指標には compute() メソッドがある。

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
print(accuracy.description)
print(accuracy.conpute(references=[0, 1, 0, 1], predictions=[1, 0, 0, 1]))

In [ ]:
f1_score = evaluate.load("f1")

def conpute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)

  # 精度と F1 スコアを求める
  acc_result = accuracy.compute(references=labels, predictions=preds)
  acc = acc_result["accuracy"]

  f1_result = f1_score.compute(
      references=labels, predictions=preds, average="weighted"
  )
  f1 = f1_result["f1"]

  return {"accuracy": acc, "f1": f1}